# Module 3 — Lab

The lesson measured six engines for you. This lab is the other way round: you take
the measurements, and the exercise each time is **reading** them.

**Every measurement here has a trap in it.** Four exercises, one trap each:

| | what you measure | the trap |
|---|---|---|
| 1 | a question of your own, on all six | a latency whose answer is wrong |
| 2 | the same Cypher text on three engines | "portable" is a claim, not a property |
| 3 | an algorithm no engine here has | the cost of leaving is not the wire |
| 4 | a write loop against a write batch | the engine is not what makes it slow |

**How to work.** Each exercise gives you the question, the tools, and the shape of
the answer. The query is yours to write. Each one ends in a number you can defend —
write the number down, and write down what it was measured against.

Budget: 1.5–2.5 hours.

**Before you start**, all four servers have to be up and the two embedded stores
loaded. The next cell checks, and tells you what is missing rather than failing on
the first query.

In [ ]:
# The same `ask` the lesson uses: one call for six engines, rows of lists back.
# Two places this notebook can run from -- the module folder (the lab) and
# solutions/ one level down -- and the tools sit at the same place relative to
# the repository root from both.
import json, os, statistics, sys, time
from pathlib import Path

TOOLS = next(p for p in (Path("../../tools/03-where-data-lives"),
                         Path("../../../tools/03-where-data-lives")) if p.exists())
sys.path.insert(0, str(TOOLS))
from engines import ask, one, up, dialect, ENGINES, LPG, RDF, EMBEDDED, GRAPH, IMPORT
from queries import cypher_for, norm

EXPECTED = {**{e: 2_137_944 for e in LPG}, **{e: 36_840_344 for e in RDF}}
HOW = {"neo4j": "Bolt, port 7690", "falkordb": "Redis protocol, port 6379",
       "kuzu": "in-process, read-only", "graphdb": "SPARQL over HTTP, port 7200",
       "fuseki": "SPARQL over HTTP, port 3030", "oxigraph": "in-process, read-only"}

down = {e: why for e, why in up().items() if why}
print(f"{'engine':<11}{'holds':>14}   how you reach it")
for e in ENGINES:
    if e in down:
        print(f"{e:<11}{'DOWN':>14}   {down[e]}")
        continue
    n = int(one(e, "MATCH (n) RETURN count(n)") if e in LPG else
            one(e, "SELECT (COUNT(*) AS ?n) WHERE { GRAPH <%s> {?s ?p ?o} }" % GRAPH))
    flag = "" if n == EXPECTED[e] else f"   <-- expected {EXPECTED[e]:,}"
    print(f"{e:<11}{n:>14,}   {HOW[e]}{flag}")
print("""
Nodes on the three property graphs, triples on the three triple stores: the two
counts are not comparable with each other, and *The trip is not free* says why.
If an engine is flagged, stop and reload it -- the rest of this lab compares
engines, and that is only meaningful while they hold the same facts.""")

## 1. A question of your own, asked of all six

The lesson's ten questions were chosen to stress one thing each — an index, a
scan, an adjacency, a join. **Add an eleventh**, and make it one the lesson does
not already answer.

Pick something real that you would want to know from this data. For example: *which
licence type appears on the most licences held by companies that also hold city
contracts?* Or: *how many organisations share an address with another organisation
in a different OrganizationGroup?* Yours is better than either if it interests you.

Write it twice — Cypher and SPARQL — and ask all six.

**Produce:** a table with one row per engine, three columns: the answer, the median
of ten runs, and whether that answer equals the others. Then the two sentences that
matter: **which engine won, and why that is a property of your question rather than
of the engine.**

**Where to look.** *The same question, six times* does this for ten questions, and
its cell is the loop to copy. `tools/03-where-data-lives/queries.py` holds the ten
texts — a dict with `cypher`, `sparql` and a `stress` saying what the question
costs — and the `norm()` that makes two engines' answers comparable.

**The trap.** You will be tempted to write the query, read the times, and report
the winner. Do not. Check the answers first and expect a disagreement: the lesson's
answer check caught one, Q9, and it was the engines. The one you are likelier to
meet is a defect in the question.

That defect is a pattern that constrains less than the question does: a SPARQL
triple pattern with no `?x rdf:type voc:Label`, or a Cypher `MATCH (x)` with no
label. **Both languages allow it and neither warns** — on this data either one
answers 25,644 where the question wanted 2,957. It is not a property of one
paradigm, it is a question you did not finish writing.

**What the exercise really is:** deciding whether a gap you measured is the engine
or your query. One of those is worth reporting.

In [ ]:
# 1. Your eleventh question. Fill in both texts, then run.
MY_QUESTION = dict(
    id="Q11_mine",
    stress="...what does this question cost? one line.",
    cypher="""
        // your Cypher
    """,
    sparql="""
        SELECT ... WHERE { GRAPH <%s> {
          # your SPARQL
        } }
    """ % GRAPH,
)
ITERATIONS, WARMUP = 10, 1

# TODO: for every engine not in `down`, run the text WARMUP times and throw the
#       result away, then ITERATIONS more, timed, and keep the median.
#       `cypher_for(MY_QUESTION, engine)` is the text to send a property graph.
#       Compare `norm(rows)` across engines BEFORE you read a single timing.

## 2. The same Cypher text, on three engines that all speak Cypher

*The same question, six times* found one question where Neo4j and FalkorDB
answered 18,670 and Kuzu answered 18,898, on identical data and identical query
text. The cause was trail semantics against walk semantics in a variable-length
path: two engines refuse to reuse a relationship inside one path, one allows it.

**Find a second divergence of your own.** Variable-length paths are the obvious
place, but they are not the only one — look also at `shortestPath`, at `DISTINCT`
over a path, at ordering with ties, at how each engine prints a double, at what
each does with a null inside an aggregate, and at the type each loader gave a
value.

**Produce:** one query text, three answers, and an explanation that accounts for the
difference **numerically** — not "they implement it differently" but *this many
rows, for this reason, and here is the arithmetic*. The lesson's example closed
exactly: the gap was 228, which is the number of edges whose reverse also exists,
each one yielding one walk that repeats an edge.

**Where to look.** *The same question, six times*, the cell that closes Q9. Note how
the explanation is built: split the answer by path length, find which length
differs, then find a count in the data that equals the difference.

**The trap.** Kuzu stores `IS_SIMILAR_TO` and `BELONGS_TO_ORG_GROUP` as one table per
label pair, so the plain name does not exist there. `ask` rewrites the plain name
for the shapes the lesson uses — `dialect(text)` shows you exactly what Kuzu was
sent — and any other shape fails with `Binder exception: Table IS_SIMILAR_TO does
not exist`. Read that as the cost of the engine's DDL, not as a bug in your query,
and do not "fix" it by dropping the relationship type: an untyped pattern runs
everywhere and answers a different question.

**What the exercise really is:** producing an explanation that closes to the row.
An unexplained divergence is not a finding, it is a thing you noticed.

In [ ]:
# 2. One text, three engines. Start from a variable-length path if you like.
Q = "MATCH p = (a:Organization)-[:IS_SIMILAR_TO*2..3]->(b:Organization) RETURN count(p) AS n"

# TODO: run Q on each of LPG and compare. `dialect(Q)` is what Kuzu receives.
#       Then split the answer until one part differs, and find the count in the
#       data that equals the difference.

## 3. An algorithm none of the six has, and what leaving costs

*Where the computation lives* showed that the widest algorithm library available
is networkx, which is not a database, and that the toll for using it is flat: the
same rows leave the store whichever algorithm follows.

**Pick an algorithm no engine here offers** and run it on the organisation
similarity graph. `nx.articulation_points` is a good choice: nothing here has it,
and it answers a question worth asking — *which single company holds a cluster
together?* Betweenness on the whole graph is another: GDS has it, FalkorDB samples
it, and the lesson showed what that costs.

**Produce:** a table with four rows — the four ways out, Neo4j and the three triple
stores — each with `export`, `compute`, and the share of the total spent moving
rather than computing. Then the sentence that section hands over, in your own
numbers: **at what compute time does the export stop mattering?**

**Where to look.** *Where the computation lives*, under *The transfer, priced*: the
cell above that heading pulls the edge list out of all three triple stores.

**The trap, and it is the most useful thing in this lab.** An edge list cannot
carry a node that has no edges, and **189,238 of the 210,740 organisations have
none**. Pull the edges, build the graph, count the components, and you get 8,465
where the right answer is 197,703 — wrong by twenty-three times, entirely
plausible-looking, and nothing warns you. The isolated nodes have to be added
back by hand. This is not RDF's trap: it belongs to every path that takes data
out of a database, which includes every time you reach for pandas.

And one that is RDF's: the exporter percent-encodes the key in the URI, so
`contracts:10598723` arrives as `.../Organization/contracts%3A10598723`. Compare
that with an id from Neo4j and nothing matches — silently.

**What the exercise really is:** knowing what your export dropped. Measure it
both ways — with the isolated nodes and without — and report both numbers.

In [ ]:
# 3. Out of the engine, into networkx, and back with a number.
import urllib.parse
import networkx as nx

EDGES_SPARQL = """SELECT ?a ?b WHERE { GRAPH <%s> {
    ?a voc:IS_SIMILAR_TO ?b . ?a rdf:type voc:Organization . ?b rdf:type voc:Organization
} }""" % GRAPH
TOTAL_ORGS = 210_740

# TODO: for Neo4j and each triple store, time the export of the edge list and,
#       separately, the computation. Report the answer from the edges alone and
#       the answer after adding back every organisation -- including the ones
#       with no edge, which the edge list cannot carry.
#
#       From the triple stores, turn each URI back into the key the property
#       graphs use: urllib.parse.unquote(uri.rsplit("/", 1)[-1]).

## 4. A write loop against a write batch

Every loader in this repository writes in batches, and the lesson read why from a
record: 2,000 rows written one at a time cost about 35 s on Fuseki and 0.02 s on
embedded Oxigraph — the same query language, over a thousand times apart.

**Reproduce it on two engines of your choice**, one served and one embedded, and
then project it.

**Produce:** for each engine, the time for N rows written singly, the time for the
same N in one statement, and the ratio. Then the projection: at that per-row rate,
**how long would the real load take row by row?** The payload is 4,073,458 edges.
Compare your projection with the batched load time in *The six engines*, which
`load-times.json` holds.

**Where to look.** The batching cell under *How the data got in* shows the two
shapes of statement. Write to a throwaway label or named graph — `BatchTest`,
`<http://kgtp.org/graph/batchtest>` — never to `kgtp`. The two embedded stores are
held read-only by this notebook, so for an embedded engine open a **new store in
a temporary directory**: same engine, same data structure on disk, and the loaded
one untouched. Fuseki takes SPARQL Update at `http://localhost:3030/ds/update`,
GraphDB at `http://localhost:7200/repositories/kgtp/statements`.

**The trap.** You will conclude that the wire is the cost. It is not, mostly.
In the lesson's record the two embedded engines come out at 3x and 14x, the two
served property graphs at 35x and 41x, and the two served triple stores at
hundreds — and GraphDB and Fuseki speak the **same protocol** as each other, on
the same machine, as Neo4j does over Bolt. What separates them is a durable
commit per statement, which a per-triple `INSERT DATA` pays every time.

**What the exercise really is:** separating two costs that arrive added together.
Your ratio has a round trip and a commit in it, and the only way to tell them
apart is to measure something that has one and not the other.

In [ ]:
# 4. One at a time, then in one statement. Use a THROWAWAY label or graph.
TEST_LABEL = "BatchTest"                        # for the property graphs
TEST_GRAPH = "http://kgtp.org/graph/batchtest"  # for the triple stores
N = 2000

# TODO: for one served engine and one embedded engine (in a temporary
#       directory), write N rows singly and then in one statement, and clean up
#       after yourself.
#
#       Then project: 4,073,458 edges at your measured per-row rate is how long?
#       Put that next to the batched load time in load-times.json.

## Close

Four numbers, and for each one the thing it was measured against. If you wrote
down only the numbers, go back: a latency with no answer beside it, a ratio with no
control, and a component count with no note about what the export dropped are all
the same mistake in different clothes.

The one sentence to carry out of this lab: **a measurement you cannot say what it
was compared with is not a measurement.**